In [1]:
import pandas as pd
import numpy as np
import mibian
import os

# Load merged 5-minute data
df = pd.read_csv(
    "../data/nifty_merged_5min.csv",
    index_col=0,
    parse_dates=True
)

# Greeks calculation function (mibian-correct)

def get_greeks(row):
    try:
        underlying = float(row["Close"])
        strike = float(row["ATM_Strike"])
        interest_rate = 6.5          # annual % rate
        days_to_expiry = 5           # approx for 5-min data

        iv_call = float(row["IV_CE"]) * 100
        iv_put = float(row["IV_PE"]) * 100

        # Call option Greeks
        call = mibian.BS(
            underlying,
            strike,
            interest_rate,
            days_to_expiry,
            volatility=iv_call
        )

        # Put option Greeks
        put = mibian.BS(
            underlying,
            strike,
            interest_rate,
            days_to_expiry,
            volatility=iv_put
        )

        return pd.Series({
            "Delta_CE": call.callDelta,
            "Gamma": call.gamma,
            "Theta_CE": call.callTheta,
            "Vega": call.vega,
            "Delta_PE": put.putDelta,
            "Theta_PE": put.putTheta
        })

    except Exception:
        return pd.Series({
            "Delta_CE": np.nan,
            "Gamma": np.nan,
            "Theta_CE": np.nan,
            "Vega": np.nan,
            "Delta_PE": np.nan,
            "Theta_PE": np.nan
        })


# Calculate Greeks

print("Calculating Greeks... This may take a moment.")
greeks_df = df.apply(get_greeks, axis=1)
df = pd.concat([df, greeks_df], axis=1)

# Derived features

df["Avg_IV"] = (df["IV_CE"] + df["IV_PE"]) / 2
df["IV_Spread"] = df["IV_CE"] - df["IV_PE"]
df["PCR_OI"] = df["OI_PE"] / df["OI_CE"]

df["Futures_Basis"] = (df["Close_fut"] - df["Close"]) / df["Close"]

df["Delta_Neutral_Ratio"] = (
    abs(df["Delta_CE"]) / abs(df["Delta_PE"])
)

df["Gamma_Exposure"] = (
    df["Close"] *
    df["Gamma"] *
    (df["OI_CE"] + df["OI_PE"])
)

# Save final feature set

os.makedirs("../data", exist_ok=True)

df.to_csv("../data/nifty_features_5min.csv")
print("Task 2 complete. File saved as nifty_features_5min.csv")


Calculating Greeks... This may take a moment.


Task 2 complete. File saved as nifty_features_5min.csv
